In [2]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [3]:
urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "ml-100k.zip")
with zipfile.ZipFile("ml-100k.zip", "r") as z:
    z.extractall(".")

In [4]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"],
    usecols=["user_id", "movie_id", "rating"]
)

movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    names=["movie_id", "title"] + [f"col_{i}" for i in range(22)],
    usecols=["movie_id", "title"]
)

#ratings.head()
#movies.head()

In [8]:
matrix = ratings.pivot_table(
    index="user_id", columns="movie_id", values="rating"
).fillna(0)

model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=1)
model.fit(matrix.values)

#print(model)
#print(matrix)

# Test profile: use 10 real ratings from an existing MovieLens user
test_user_id = 1

sample = ratings[ratings["user_id"] == test_user_id].head(10)
mock_user_ratings = dict(zip(sample["movie_id"], sample["rating"]))

mock_user_ratings

{61: 4, 189: 3, 33: 4, 160: 4, 20: 4, 202: 5, 171: 5, 265: 4, 155: 2, 117: 3}

In [18]:
# KNN model

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import fetch_openml
import pandas as pd

# load data

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"],
    usecols=["user_id", "movie_id", "rating"]
)

# pivot the table so the index is user id for fitting into knn

user_movie_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
).fillna(0) # 0 for no ratings

# print(f"Matrix shape: {user_movie_matrix.shape} ") # should be shape users x movies

# knn model

model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=1) # https://scikit-learn.org/stable/modules/neighbors.html
model.fit(user_movie_matrix)

# convert to match input dimensions
# 0 for all movies not rated

user_vector = pd.Series(mock_user_ratings).reindex(
    user_movie_matrix.columns, fill_value=0
)

# return matching user

neighbors = 1  # only need top 1 match for now

distances, indices = model.kneighbors(
    user_vector.values.reshape(1, -1),
    n_neighbors=neighbors
)

indices  = indices[0] # remove apparent nested array
distances = distances[0]

matched_users = []
for i in range(len(indices)):
    user_id    = user_movie_matrix.index[indices[i]]
    similarity = 1 - distances[i]   # convert distance to similarity
    matched_users.append({"user_id": user_id, "similarity": similarity})

best_match = matched_users[0]
print(best_match['user_id'])

1


In [ ]:
# final_movie_recommendation section

def find_best_user(mock_user_ratings, user_movie_matrix, model):
    cleaned_ratings = {
        int(movie_id): rating for movie_id, rating in mock_user_ratings.items()
    }

    user_vector = pd.Series(cleaned_ratings, dtype=float).reindex(
        user_movie_matrix.columns,
        fill_value=0,
    )

    distances, indices = model.kneighbors(
        user_vector.values.reshape(1, -1),
        n_neighbors=1,
    )

    return {
        "user_id": int(user_movie_matrix.index[indices[0][0]]),
        "similarity": float(1 - distances[0][0]),
    }


def build_used_movies_dict(shown_movie_ids, movie_lookup, mock_user_ratings):
    cleaned_ratings = {
        int(movie_id): rating for movie_id, rating in mock_user_ratings.items()
    }

    return {
        int(movie_id): {
            "title": movie_lookup.get(int(movie_id), f"Movie {movie_id}"),
            "user_rating": cleaned_ratings.get(int(movie_id)),
        }
        for movie_id in shown_movie_ids
    }


def build_recommendation_pool(
    best_user_id,
    ratings,
    movie_lookup,
    excluded_movie_ids,
    random_state=None,
):
    rng = np.random.default_rng(random_state)

    candidate_movies = ratings[ratings["user_id"] == best_user_id].copy()
    candidate_movies = candidate_movies[
        ~candidate_movies["movie_id"].isin(excluded_movie_ids)
    ].drop_duplicates(subset="movie_id")

    ranked_groups = []
    for rating_value in sorted(candidate_movies["rating"].unique(), reverse=True):
        rating_group = candidate_movies[
            candidate_movies["rating"] == rating_value
        ].copy()
        if len(rating_group) > 1:
            rating_group = rating_group.iloc[rng.permutation(len(rating_group))]
        ranked_groups.append(rating_group)

    if ranked_groups:
        candidate_movies = pd.concat(ranked_groups, ignore_index=True)

    candidate_movies["title"] = candidate_movies["movie_id"].map(movie_lookup)

    return candidate_movies[["movie_id", "title", "rating"]].to_dict("records")


def start_recommendation_session(
    mock_user_ratings,
    user_movie_matrix,
    model,
    ratings,
    movies,
    shown_movie_ids=None,
    random_state=None,
):
    if shown_movie_ids is None:
        shown_movie_ids = list(mock_user_ratings.keys())

    shown_movie_ids = [int(movie_id) for movie_id in shown_movie_ids]
    movie_lookup = movies.set_index("movie_id")["title"].to_dict()

    best_user = find_best_user(mock_user_ratings, user_movie_matrix, model)
    used_movies = build_used_movies_dict(
        shown_movie_ids,
        movie_lookup,
        mock_user_ratings,
    )

    recommendation_pool = build_recommendation_pool(
        best_user_id=best_user["user_id"],
        ratings=ratings,
        movie_lookup=movie_lookup,
        excluded_movie_ids=set(shown_movie_ids),
        random_state=random_state,
    )

    return {
        "best_user": best_user,
        "used_movies": used_movies,
        "remaining_recommendations": recommendation_pool,
    }


def get_next_recommendations(session, group_size=3):
    batch = session["remaining_recommendations"][:group_size]
    session["remaining_recommendations"] = session["remaining_recommendations"][group_size:]
    return pd.DataFrame(batch)


recommendation_session = start_recommendation_session(
    mock_user_ratings=mock_user_ratings,
    user_movie_matrix=user_movie_matrix,
    model=model,
    ratings=ratings,
    movies=movies,
    random_state=42,
)

best_user = recommendation_session["best_user"]
used_movies = recommendation_session["used_movies"]

print("Best matching user:", best_user["user_id"])
print("Similarity score:", round(best_user["similarity"], 4))
print("Movies to ignore when recommending:")
print(used_movies)

first_recommendations = get_next_recommendations(recommendation_session, group_size=3)
print("\nFirst 3 recommendations:")
display(first_recommendations)


Best matching user: 1
Similarity score: 0.1955
Movies to ignore when recommending:
{61: {'title': 'Three Colors: White (1994)', 'user_rating': 4}, 189: {'title': 'Grand Day Out, A (1992)', 'user_rating': 3}, 33: {'title': 'Desperado (1995)', 'user_rating': 4}, 160: {'title': 'Glengarry Glen Ross (1992)', 'user_rating': 4}, 20: {'title': 'Angels and Insects (1995)', 'user_rating': 4}, 202: {'title': 'Groundhog Day (1993)', 'user_rating': 5}, 171: {'title': 'Delicatessen (1991)', 'user_rating': 5}, 265: {'title': 'Hunt for Red October, The (1990)', 'user_rating': 4}, 155: {'title': 'Dirty Dancing (1987)', 'user_rating': 2}, 117: {'title': 'Rock, The (1996)', 'user_rating': 3}}

First 3 recommendations:


,movie_id,title,rating
0,9,Dead Man Walking (1995),5
1,169,"Wrong Trousers, The (1993)",5
2,270,Gattaca (1997),5


In [16]:
more_recommendations = get_next_recommendations(recommendation_session, group_size=3)
display(more_recommendations)

,movie_id,title,rating
0,12,"Usual Suspects, The (1995)",5
1,177,"Good, The Bad and The Ugly, The (1966)",5
2,190,Henry V (1989),5
